Cell 1 — Mount Drive and load the homology-aware data

In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:

import os
import json
import math
import random
import yaml
import numpy as np
import pandas as pd
import torch

from scipy import stats
from sklearn.metrics import average_precision_score
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm

PROJECT_ROOT = "/content/drive/MyDrive/atlas-go-revision"
os.chdir(PROJECT_ROOT)

with open("configs/config.yaml", "r") as file:
    cfg = yaml.safe_load(file)

PROCESSED_DIR = cfg["paths"]["data_processed"]
RESULTS_DIR = cfg["paths"]["results"]
MODELS_DIR = cfg["paths"]["models"]

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Y_all = np.load(f"{PROCESSED_DIR}/Y_all.npy").astype(np.float32)

splits = np.load(f"{PROCESSED_DIR}/splits_homology.npz")
train_idx = splits["train_idx"]
val_idx = splits["val_idx"]
test_idx = splits["test_idx"]

go_map = pd.read_csv(f"{PROCESSED_DIR}/go_namespace_map.csv")
go_ids = go_map["go_id"].tolist()

with open(f"{PROCESSED_DIR}/go_ontology_slices.yaml", "r") as file:
    ontology_slices = yaml.safe_load(file)

print("Proteins:", len(Y_all))
print("GO terms:", Y_all.shape[1])
print("Train / validation / test:", len(train_idx), len(val_idx), len(test_idx))
print("Training seeds:", cfg["training_seeds"])

Device: cpu
Proteins: 1356
GO terms: 1961
Train / validation / test: 1091 133 132
Training seeds: [42, 1, 7]


Cell 2 — Reproducibility function

In [6]:
def set_seed(seed):
    """Make one training run reproducible."""
    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(cfg["split"]["random_seed_for_split"])
print("Reproducibility settings applied.")

Reproducibility settings applied.


In [7]:
!pip install goatools

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 57.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 11.4 MB/s eta 0:00:00


Cell 3 — GO hierarchy score propagation

In [8]:
from goatools.obo_parser import GODag

obo_path = cfg["go_hierarchy"]["obo_path"]
go_dag = GODag(obo_path, optional_attrs={"relationship"})

go_to_index = {go_id: index for index, go_id in enumerate(go_ids)}

# For every retained GO term, find retained ancestor terms.
ancestor_indices = {}

for go_id in go_ids:
    ancestors = set()
    to_visit = [go_id]
    visited = set()

    while to_visit:
        current_id = to_visit.pop()

        if current_id in visited or current_id not in go_dag:
            continue

        visited.add(current_id)
        term = go_dag[current_id]

        parent_ids = [parent.item_id for parent in term.parents]

        for parent in term.relationship.get("part_of", set()):
            parent_ids.append(parent.item_id)

        for parent_id in parent_ids:
            if parent_id not in ancestors:
                ancestors.add(parent_id)
                to_visit.append(parent_id)

    ancestor_indices[go_to_index[go_id]] = [
        go_to_index[parent_id]
        for parent_id in ancestors
        if parent_id in go_to_index
    ]

with open(f"{PROCESSED_DIR}/retained_go_ancestor_map.json", "w") as file:
    json.dump(
        {
            go_ids[index]: [go_ids[parent] for parent in parents]
            for index, parents in ancestor_indices.items()
        },
        file,
        indent=2,
    )

print("GO ancestor map created.")

def propagate_prediction_scores(probabilities):
    """
    Apply the GO true-path rule to prediction scores.

    If a child term has a high prediction score, every retained ancestor
    receives at least that score.
    """
    propagated = probabilities.copy()

    for child_index, parent_indexes in ancestor_indices.items():
        if parent_indexes:
            propagated[:, parent_indexes] = np.maximum(
                propagated[:, parent_indexes],
                propagated[:, [child_index]],
            )

    return propagated

/content/drive/MyDrive/atlas-go-revision/data/raw/go-basic.obo: fmt(1.2) rel(2025-10-10) 42,666 Terms; optional_attrs(relationship)
GO ancestor map created.


Cell 4 — CAFA Fmax and AUPRC metrics

In [9]:
def cafa_fmax(probabilities, labels, thresholds=None):
    """
    CAFA protein-centric Fmax.

    Precision is averaged only across proteins receiving at least one prediction.
    Recall is averaged across all proteins.
    """
    if thresholds is None:
        thresholds = np.arange(0.01, 1.00, 0.01)

    best_fmax = 0.0
    best_threshold = 0.0
    best_precision = 0.0
    best_recall = 0.0

    labels_bool = labels.astype(bool)
    true_count_per_protein = labels_bool.sum(axis=1)

    for threshold in thresholds:
        predicted = probabilities >= threshold

        predicted_count_per_protein = predicted.sum(axis=1)
        true_positive_per_protein = (
            predicted & labels_bool
        ).sum(axis=1)

        proteins_with_predictions = predicted_count_per_protein > 0

        if proteins_with_predictions.any():
            precision = np.mean(
                true_positive_per_protein[proteins_with_predictions]
                / predicted_count_per_protein[proteins_with_predictions]
            )
        else:
            precision = 0.0

        recall = np.mean(
            true_positive_per_protein
            / np.maximum(true_count_per_protein, 1)
        )

        if precision + recall > 0:
            f_score = 2 * precision * recall / (precision + recall)
        else:
            f_score = 0.0

        if f_score > best_fmax:
            best_fmax = float(f_score)
            best_threshold = float(threshold)
            best_precision = float(precision)
            best_recall = float(recall)

    return {
        "Fmax": best_fmax,
        "best_threshold": best_threshold,
        "precision_at_Fmax": best_precision,
        "recall_at_Fmax": best_recall,
    }


def cafa_f_at_threshold(probabilities, labels, threshold):
    """Protein-centric CAFA F-score at one fixed threshold."""
    labels_bool = labels.astype(bool)
    predicted = probabilities >= threshold

    predicted_count = predicted.sum(axis=1)
    true_positive = (predicted & labels_bool).sum(axis=1)
    true_count = labels_bool.sum(axis=1)

    proteins_with_predictions = predicted_count > 0

    if proteins_with_predictions.any():
        precision = np.mean(
            true_positive[proteins_with_predictions]
            / predicted_count[proteins_with_predictions]
        )
    else:
        precision = 0.0

    recall = np.mean(true_positive / np.maximum(true_count, 1))

    if precision + recall == 0:
        return 0.0

    return float(2 * precision * recall / (precision + recall))


def macro_auprc(probabilities, labels):
    """
    Macro AUPRC over GO terms having both positive and negative test examples.
    """
    scores = []

    for term_index in range(labels.shape[1]):
        y_true = labels[:, term_index]

        if y_true.sum() == 0 or y_true.sum() == len(y_true):
            continue

        scores.append(
            average_precision_score(y_true, probabilities[:, term_index])
        )

    return float(np.mean(scores)) if scores else np.nan


def evaluate_predictions(probabilities, labels):
    """Compute overall and per-ontology metrics, excluding GO root terms."""
    if cfg["go_hierarchy"]["propagate_predictions"]:
        probabilities = propagate_prediction_scores(probabilities)

    output = {}

    # Overall metrics: score only non-root terms.
    overall = cafa_fmax(
        probabilities[:, scored_term_mask],
        labels[:, scored_term_mask],
    )
    output["Overall_Fmax"] = overall["Fmax"]
    output["Overall_best_threshold"] = overall["best_threshold"]
    output["Overall_AUPRC"] = macro_auprc(
        probabilities[:, scored_term_mask],
        labels[:, scored_term_mask],
    )

    for ontology_name in ["MFO", "BPO", "CCO"]:
        start = ontology_slices[ontology_name]["start"]
        end = ontology_slices[ontology_name]["end"]

        ontology_mask = scored_term_mask[start:end]

        ontology_probabilities = probabilities[:, start:end][:, ontology_mask]
        ontology_labels = labels[:, start:end][:, ontology_mask]

        ontology_result = cafa_fmax(
            ontology_probabilities,
            ontology_labels,
        )

        output[f"{ontology_name}_Fmax"] = ontology_result["Fmax"]
        output[f"{ontology_name}_AUPRC"] = macro_auprc(
            ontology_probabilities,
            ontology_labels,
        )

    return output




In [10]:
# --- Root-term exclusion (CAFA standard practice) ---
# These three GO terms are the top of each ontology hierarchy.
# After true-path propagation they are present in the vast majority
# of proteins, making them trivially predictable and inflating any
# method's score, including the naive baseline. CAFA excludes them
# from scoring; we do the same here.

ROOT_TERMS = {"GO:0003674", "GO:0008150", "GO:0005575"}

root_term_indexes = [
    index for index, go_id in enumerate(go_ids) if go_id in ROOT_TERMS
]

scored_term_mask = np.ones(len(go_ids), dtype=bool)
scored_term_mask[root_term_indexes] = False

print("Root terms found in vocabulary:", len(root_term_indexes))
print("Terms retained for scoring:", scored_term_mask.sum(), "of", len(go_ids))

Root terms found in vocabulary: 3
Terms retained for scoring: 1958 of 1961


Cell 5 — Continuous training-frequency baseline

In [8]:
# Every GO term receives its frequency in the training set.
# Every test protein receives the same continuous score vector.

training_label_frequency = Y_all[train_idx].mean(axis=0)

baseline_probabilities = np.tile(
    training_label_frequency,
    (len(test_idx), 1),
)

baseline_results = evaluate_predictions(
    probabilities=baseline_probabilities,
    labels=Y_all[test_idx],
)

baseline_results["Architecture"] = "Frequency baseline"
baseline_results["Feature"] = "All GO-term training frequencies"
baseline_results["seed"] = cfg["split"]["random_seed_for_split"]

baseline_table = pd.DataFrame([baseline_results])

baseline_table.to_csv(
    f"{RESULTS_DIR}/continuous_frequency_baseline_homology.csv",
    index=False,
)

print("Continuous frequency baseline")
print("-" * 50)

for key, value in baseline_results.items():
    if isinstance(value, float):
        print(f"{key}: {value:.4f}")

Continuous frequency baseline
--------------------------------------------------
Overall_Fmax: 0.3583
Overall_best_threshold: 0.1600
Overall_AUPRC: 0.0372
MFO_Fmax: 0.3807
MFO_AUPRC: 0.0477
BPO_Fmax: 0.2577
BPO_AUPRC: 0.0326
CCO_Fmax: 0.3930
CCO_AUPRC: 0.0445


Cell 6 — Test-set GO-term prevalence report

In [11]:
test_prevalence = go_map.copy()

test_prevalence["n_test_proteins"] = (
    Y_all[test_idx].sum(axis=0).astype(int)
)

test_prevalence["test_prevalence"] = (
    test_prevalence["n_test_proteins"] / len(test_idx)
)

test_prevalence = test_prevalence.sort_values(
    ["namespace", "n_test_proteins"],
    ascending=[True, False],
)

test_prevalence.to_csv(
    f"{RESULTS_DIR}/test_term_prevalence_homology.csv",
    index=False,
)

print(test_prevalence.groupby("namespace")["n_test_proteins"].describe())

            count      mean        std  min  25%  50%  75%    max
namespace                                                        
BPO        1350.0  2.808148   6.560169  0.0  0.0  1.0  3.0  106.0
CCO         267.0  4.501873  11.698716  0.0  0.0  1.0  3.0  104.0
MFO         344.0  4.808140  11.989468  0.0  0.0  2.0  3.0  119.0


Cell 7 — Define the three neural-network architectures

In [12]:
class FocalLoss(nn.Module):
    """Focal loss for imbalanced multi-label GO prediction."""

    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        bce = nn.functional.binary_cross_entropy_with_logits(
            logits,
            targets,
            reduction="none",
        )

        probabilities = torch.sigmoid(logits)
        p_t = probabilities * targets + (1 - probabilities) * (1 - targets)

        focal_weight = (1 - p_t) ** self.gamma

        # Higher weight for positive labels; lower weight for negatives.
        alpha_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)

        return (alpha_t * focal_weight * bce).mean()




class MLP(nn.Module):
    """Original four-layer MLP architecture."""

    def __init__(self, input_dim, num_classes):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_dim, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(1024, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(512, 256),
            nn.ReLU(),

            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        return self.network(x)


class CNN1D(nn.Module):
    """
    Original CNN1D idea:
    treat the concatenated feature vector as a one-channel signal.
    """

    def __init__(self, input_dim, num_classes):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv1d(1, 128, kernel_size=7),
            nn.BatchNorm1d(128),
            nn.ReLU(),

            nn.Conv1d(128, 64, kernel_size=5),
            nn.BatchNorm1d(64),
            nn.ReLU(),

            nn.Conv1d(64, 32, kernel_size=3),
            nn.BatchNorm1d(32),
            nn.ReLU(),
        )

        # Three convolution layers reduce length by 6 + 4 + 2 = 12.
        conv_output_length = input_dim - 12

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * conv_output_length, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes),
        )

    def forward(self, x):
        x = x.unsqueeze(1)
        x = self.features(x)
        return self.classifier(x)


class ResidualFFNBlock(nn.Module):
    """One residual feed-forward block."""

    def __init__(self, hidden_dim, dropout):
        super().__init__()

        self.layer_norm = nn.LayerNorm(hidden_dim)

        self.feed_forward = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return x + self.feed_forward(self.layer_norm(x))


class ResFFN(nn.Module):
    """
    Renamed version of the previous TransformerGO architecture.

    It contains residual feed-forward blocks, not self-attention.
    """

    def __init__(self, input_dim, num_classes):
        super().__init__()

        settings = cfg["training"]["resffn"]
        hidden_dim = settings["hidden_dim"]
        num_layers = settings["num_layers"]
        dropout = settings["dropout"]

        self.input_projection = nn.Linear(input_dim, hidden_dim)

        self.blocks = nn.Sequential(
            *[
                ResidualFFNBlock(hidden_dim, dropout)
                for _ in range(num_layers)
            ]
        )

        self.output_norm = nn.LayerNorm(hidden_dim)
        self.output_layer = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        x = self.input_projection(x)
        x = self.blocks(x)
        x = self.output_norm(x)
        return self.output_layer(x)


ARCHITECTURES = {
    "MLP": MLP,
    "CNN1D": CNN1D,
    "ResFFN": ResFFN,
}

print("Architectures:", list(ARCHITECTURES.keys()))

Architectures: ['MLP', 'CNN1D', 'ResFFN']


Cell 8 — Training and prediction functions

In [14]:
def make_data_loader(features, labels, batch_size, shuffle, seed):
    generator = torch.Generator()
    generator.manual_seed(seed)

    dataset = TensorDataset(
        torch.tensor(features, dtype=torch.float32),
        torch.tensor(labels, dtype=torch.float32),
    )

    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=0,
        generator=generator,
    )


def predict_probabilities(model, features, batch_size=128):
    """Predict probabilities without loading all samples onto GPU at once."""
    model.eval()

    probabilities = []

    with torch.no_grad():
        for start in range(0, len(features), batch_size):
            batch = torch.tensor(
                features[start:start + batch_size],
                dtype=torch.float32,
                device=device,
            )

            logits = model(batch)
            probabilities.append(torch.sigmoid(logits).cpu().numpy())

    return np.vstack(probabilities)


def get_architecture_settings(architecture_name):
    if architecture_name == "MLP":
        return cfg["training"]["mlp"]

    if architecture_name == "CNN1D":
        return cfg["training"]["cnn"]

    if architecture_name == "ResFFN":
        return cfg["training"]["resffn"]

    raise ValueError(f"Unknown architecture: {architecture_name}")


def train_one_model(
    architecture_name,
    features,
    seed,
    learning_rate=None,
    weight_decay=None,
    save_checkpoint=False,
    checkpoint_path=None,
    evaluate_test=True,
):
    """
    Train one architecture-feature-seed run.

    Early stopping uses validation Fmax only.
    The test set is not used during training or early stopping.
    """
    set_seed(seed)

    model_class = ARCHITECTURES[architecture_name]

    input_dim = features.shape[1]
    num_classes = Y_all.shape[1]

    settings = get_architecture_settings(architecture_name)

    batch_size = settings["batch_size"]

    if learning_rate is None:
        learning_rate = settings["learning_rate"]

    if weight_decay is None:
        weight_decay = settings["weight_decay"]

    model = model_class(input_dim, num_classes).to(device)

    loss_function = FocalLoss(
        alpha=cfg["training"]["focal_loss_alpha"],
        gamma=cfg["training"]["focal_loss_gamma"],
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay,
    )

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=cfg["training"]["cosine_annealing_tmax"],
    )

    train_loader = make_data_loader(
        features[train_idx],
        Y_all[train_idx],
        batch_size=batch_size,
        shuffle=True,
        seed=seed,
    )

    best_validation_fmax = -1.0
    best_validation_threshold = 0.50
    best_state = None
    patience_counter = 0

    for epoch in range(1, cfg["training"]["max_epochs"] + 1):
        model.train()

        epoch_loss = 0.0

        for batch_features, batch_labels in train_loader:
            batch_features = batch_features.to(device)
            batch_labels = batch_labels.to(device)

            optimizer.zero_grad()

            logits = model(batch_features)
            loss = loss_function(logits, batch_labels)

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                cfg["training"]["gradient_clip_norm"],
            )

            optimizer.step()

            epoch_loss += loss.item()

        scheduler.step()

        validation_probabilities = predict_probabilities(
            model,
            features[val_idx],
        )

        validation_metrics = evaluate_predictions(
            validation_probabilities,
            Y_all[val_idx],
        )

        validation_fmax = validation_metrics["Overall_Fmax"]

        if validation_fmax > best_validation_fmax:
            best_validation_fmax = validation_fmax
            best_validation_threshold = validation_metrics[
                "Overall_best_threshold"
            ]

            best_state = {
                name: parameter.detach().cpu().clone()
                for name, parameter in model.state_dict().items()
            }

            patience_counter = 0

        else:
            patience_counter += 1

        if patience_counter >= cfg["training"]["early_stopping_patience"]:
            break
    model.load_state_dict(best_state)
    model = model.to(device)
    model.eval()

    if not evaluate_test:
        return {
            "Validation_best_Fmax": best_validation_fmax,
            "Validation_selected_threshold": best_validation_threshold,
            "epochs_trained": epoch,
        }, None

    test_probabilities = predict_probabilities(
        model,
        features[test_idx],
    )


    test_metrics = evaluate_predictions(
        test_probabilities,
        Y_all[test_idx],
    )

    # This is a secondary reproducibility metric:
    # test F-score at the threshold selected using validation data.
    propagated_test_probabilities = propagate_prediction_scores(
        test_probabilities
    )

    test_metrics["Overall_F_at_validation_threshold"] = (
        cafa_f_at_threshold(
            propagated_test_probabilities[:, scored_term_mask],
            Y_all[test_idx][:, scored_term_mask],
            best_validation_threshold,
        )
    )

    test_metrics["Validation_best_Fmax"] = best_validation_fmax
    test_metrics["Validation_selected_threshold"] = best_validation_threshold
    test_metrics["epochs_trained"] = epoch

    if save_checkpoint:
        torch.save(
            model.state_dict(),
            checkpoint_path,
        )

    del model

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return test_metrics, test_probabilities

Cell 9 — Optional validation-only hyperparameter search
Run this cell once. It uses the PES feature set and validation Fmax only.

In [15]:
RUN_HYPERPARAMETER_SEARCH = cfg["hyperparameter_search"]["enabled"]

if RUN_HYPERPARAMETER_SEARCH:
    print("Starting validation-only hyperparameter search.")

    X_tuning = np.load(
        f"{PROCESSED_DIR}/X_pes_homology.npy"
    ).astype(np.float32)

    tuning_rows = []

    for architecture_name in ARCHITECTURES:
        for learning_rate in cfg["hyperparameter_search"]["learning_rates"]:
            for weight_decay in cfg["hyperparameter_search"]["weight_decays"]:
                print(
                    f"Tuning {architecture_name} | "
                    f"lr={learning_rate} | wd={weight_decay}"
                )

                metrics, _ = train_one_model(
                    architecture_name=architecture_name,
                    features=X_tuning,
                    seed=cfg["training_seeds"][0],
                    learning_rate=learning_rate,
                    weight_decay=weight_decay,
                    evaluate_test=False,
                )

                tuning_rows.append({
                    "Architecture": architecture_name,
                    "Feature_used_for_tuning": "PES",
                    "learning_rate": learning_rate,
                    "weight_decay": weight_decay,
                    "Validation_Fmax": metrics["Validation_best_Fmax"],
                    "epochs_trained": metrics["epochs_trained"],
                })

    tuning_results = pd.DataFrame(tuning_rows)

    tuning_results.to_csv(
        f"{RESULTS_DIR}/hyperparameter_search_homology.csv",
        index=False,
    )

    best_tuning = (
        tuning_results
        .sort_values(
            ["Architecture", "Validation_Fmax"],
            ascending=[True, False],
        )
        .groupby("Architecture")
        .head(1)
    )

    best_tuning.to_csv(
        f"{RESULTS_DIR}/selected_hyperparameters_homology.csv",
        index=False,
    )

    print("\nBest validation-only settings:")
    display(best_tuning)

else:
    print(
        "Hyperparameter search is disabled in config.yaml. "
        "The predefined settings will be used."
    )

Starting validation-only hyperparameter search.
Tuning MLP | lr=0.0001 | wd=1e-05


KeyboardInterrupt: 

Cell 10 — Run all 81 training experiments

In [17]:
FEATURE_NAMES = [
    "p", "ps", "psd",
    "e", "es", "esd",
    "pe", "pes", "pesd",
]

SEEDS = cfg["training_seeds"]

results_path = f"{RESULTS_DIR}/per_seed_results_homology.csv"

if os.path.exists(results_path):
    per_seed_results = pd.read_csv(results_path)

    completed_runs = set(
        zip(
            per_seed_results["Architecture"],
            per_seed_results["Feature"],
            per_seed_results["seed"],
        )
    )

    print("Resuming previous work.")
    print("Completed runs:", len(completed_runs))

else:
    per_seed_results = pd.DataFrame()
    completed_runs = set()

for feature_name in FEATURE_NAMES:
    X = np.load(
        f"{PROCESSED_DIR}/X_{feature_name}_homology.npy"
    ).astype(np.float32)

    for architecture_name in ARCHITECTURES:
        for seed in SEEDS:
            run_key = (
                architecture_name,
                feature_name.upper(),
                seed,
            )

            if run_key in completed_runs:
                print("Skipping completed run:", run_key)
                continue

            print("\n" + "=" * 70)
            print(
                f"Training {architecture_name} | "
                f"{feature_name.upper()} | seed={seed}"
            )
            print("=" * 70)

            metrics, _ = train_one_model(
                architecture_name=architecture_name,
                features=X,
                seed=seed,
            )

            row = {
                "Architecture": architecture_name,
                "Feature": feature_name.upper(),
                "seed": seed,
                **metrics,
            }

            per_seed_results = pd.concat(
                [per_seed_results, pd.DataFrame([row])],
                ignore_index=True,
            )

            # Save after every run, so Colab disconnection does not lose progress.
            per_seed_results.to_csv(
                results_path,
                index=False,
            )

            print(
                f"Finished: Fmax={metrics['Overall_Fmax']:.4f} | "
                f"AUPRC={metrics['Overall_AUPRC']:.4f} | "
                f"epochs={metrics['epochs_trained']}"
            )

print("\nAll available runs are saved in:")
print(results_path)

Resuming previous work.
Completed runs: 81
Skipping completed run: ('MLP', 'P', 42)
Skipping completed run: ('MLP', 'P', 1)
Skipping completed run: ('MLP', 'P', 7)
Skipping completed run: ('CNN1D', 'P', 42)
Skipping completed run: ('CNN1D', 'P', 1)
Skipping completed run: ('CNN1D', 'P', 7)
Skipping completed run: ('ResFFN', 'P', 42)
Skipping completed run: ('ResFFN', 'P', 1)
Skipping completed run: ('ResFFN', 'P', 7)
Skipping completed run: ('MLP', 'PS', 42)
Skipping completed run: ('MLP', 'PS', 1)
Skipping completed run: ('MLP', 'PS', 7)
Skipping completed run: ('CNN1D', 'PS', 42)
Skipping completed run: ('CNN1D', 'PS', 1)
Skipping completed run: ('CNN1D', 'PS', 7)
Skipping completed run: ('ResFFN', 'PS', 42)
Skipping completed run: ('ResFFN', 'PS', 1)
Skipping completed run: ('ResFFN', 'PS', 7)
Skipping completed run: ('MLP', 'PSD', 42)
Skipping completed run: ('MLP', 'PSD', 1)
Skipping completed run: ('MLP', 'PSD', 7)
Skipping completed run: ('CNN1D', 'PSD', 42)
Skipping completed r

Cell 11 — Aggregate mean, SD, and 95% confidence intervals







In [18]:
per_seed_results = pd.read_csv(results_path)

metric_columns = [
    "Overall_Fmax",
    "Overall_AUPRC",
    "MFO_Fmax",
    "BPO_Fmax",
    "CCO_Fmax",
    "Overall_F_at_validation_threshold",
    "Validation_best_Fmax",
]


summary_rows = []

for (architecture_name, feature_name), group in per_seed_results.groupby(
    ["Architecture", "Feature"]
):
    row = {
        "Architecture": architecture_name,
        "Feature": feature_name,
        "n_seeds": len(group),
    }

    for metric_name in metric_columns:
        values = group[metric_name].dropna().to_numpy()

        mean_value = float(np.mean(values))
        std_value = float(np.std(values, ddof=1)) if len(values) > 1 else 0.0

        if len(values) > 1:
            standard_error = stats.sem(values)
            t_critical = stats.t.ppf(
                0.975,
                df=len(values) - 1,
            )

            ci_lower = mean_value - t_critical * standard_error
            ci_upper = mean_value + t_critical * standard_error

        else:
            ci_lower = mean_value
            ci_upper = mean_value

        row[f"{metric_name}_mean"] = mean_value
        row[f"{metric_name}_std"] = std_value
        row[f"{metric_name}_CI95_low"] = ci_lower
        row[f"{metric_name}_CI95_high"] = ci_upper

    summary_rows.append(row)

summary_results = pd.DataFrame(summary_rows)

summary_results = summary_results.sort_values(
    ["Architecture", "Overall_Fmax_mean"],
    ascending=[True, False],
).reset_index(drop=True)

summary_results.to_csv(
    f"{RESULTS_DIR}/full_results_homology_mean_sd_ci.csv",
    index=False,
)

print("Top configurations:")
display(
    summary_results[
        [
            "Architecture",
            "Feature",
            "n_seeds",
            "Overall_Fmax_mean",
            "Overall_Fmax_std",
            "Overall_Fmax_CI95_low",
            "Overall_Fmax_CI95_high",
            "Overall_AUPRC_mean",
            "Overall_AUPRC_std",
        ]
    ].head(12)
)

Top configurations:


,Architecture,Feature,n_seeds,Overall_Fmax_mean,Overall_Fmax_std,Overall_Fmax_CI95_low,Overall_Fmax_CI95_high,Overall_AUPRC_mean,Overall_AUPRC_std
0,CNN1D,ESD,3,0.439203,0.023071,0.381891,0.496516,0.219989,0.031198
1,CNN1D,E,3,0.430115,0.003936,0.420337,0.439893,0.210599,0.003967
2,CNN1D,ES,3,0.427924,0.004974,0.415569,0.440279,0.204705,0.001764
3,CNN1D,PES,3,0.423643,0.019683,0.374747,0.472539,0.203983,0.017632
4,CNN1D,PE,3,0.422670,0.017132,0.380112,0.465228,0.202211,0.020260
5,CNN1D,P,3,0.420692,0.002693,0.414002,0.427382,0.190374,0.002167
6,CNN1D,PSD,3,0.417480,0.002098,0.412268,0.422693,0.189097,0.002961
7,CNN1D,PS,3,0.414931,0.001710,0.410683,0.419179,0.188893,0.005996
8,CNN1D,PESD,3,0.405202,0.004215,0.394730,0.415673,0.188061,0.001310
9,MLP,PES,3,0.468456,0.011510,0.439864,0.497047,0.297244,0.033401


Cell 12 — Matched tests for the effect of Dynamic features

In [19]:
# Matched comparisons requested by the reviewer:
# PS vs PSD, ES vs ESD, and PES vs PESD.
# Each comparison uses the same architecture and same three training seeds.

dynamic_pairs = [
    ("PS", "PSD"),
    ("ES", "ESD"),
    ("PES", "PESD"),
]

test_rows = []

for architecture_name in ARCHITECTURES:
    architecture_data = per_seed_results[
        per_seed_results["Architecture"] == architecture_name
    ]

    for without_dynamic, with_dynamic in dynamic_pairs:
        base = architecture_data[
            architecture_data["Feature"] == without_dynamic
        ][["seed", "Overall_Fmax"]].rename(
            columns={"Overall_Fmax": "Fmax_without_dynamic"}
        )

        dynamic = architecture_data[
            architecture_data["Feature"] == with_dynamic
        ][["seed", "Overall_Fmax"]].rename(
            columns={"Overall_Fmax": "Fmax_with_dynamic"}
        )

        paired = pd.merge(base, dynamic, on="seed")

        paired["difference_dynamic_minus_no_dynamic"] = (
            paired["Fmax_with_dynamic"]
            - paired["Fmax_without_dynamic"]
        )

        differences = paired[
            "difference_dynamic_minus_no_dynamic"
        ].to_numpy()

        if len(differences) >= 2 and not np.allclose(differences, 0):
            try:
                _, p_value = stats.wilcoxon(differences)
            except ValueError:
                p_value = np.nan
        else:
            p_value = np.nan

        test_rows.append({
            "Architecture": architecture_name,
            "Without_dynamic_features": without_dynamic,
            "With_dynamic_features": with_dynamic,
            "n_seeds": len(paired),
            "mean_Fmax_without_dynamic": paired[
                "Fmax_without_dynamic"
            ].mean(),
            "mean_Fmax_with_dynamic": paired[
                "Fmax_with_dynamic"
            ].mean(),
            "mean_difference_dynamic_minus_no_dynamic": differences.mean(),
            "std_difference": (
                differences.std(ddof=1)
                if len(differences) > 1 else 0.0
            ),
            "Wilcoxon_p_value": p_value,
        })

dynamic_tests = pd.DataFrame(test_rows)

dynamic_tests.to_csv(
    f"{RESULTS_DIR}/dynamic_feature_matched_tests_homology.csv",
    index=False,
)

display(dynamic_tests)

,Architecture,Without_dynamic_features,With_dynamic_features,n_seeds,mean_Fmax_without_dynamic,mean_Fmax_with_dynamic,mean_difference_dynamic_minus_no_dynamic,std_difference,Wilcoxon_p_value
0,MLP,PS,PSD,3,0.430580,0.429496,-0.001085,0.013405,1.00
1,MLP,ES,ESD,3,0.458969,0.445434,-0.013535,0.012180,0.25
2,MLP,PES,PESD,3,0.468456,0.455120,-0.013335,0.008487,0.25
3,CNN1D,PS,PSD,3,0.414931,0.417480,0.002549,0.001463,0.25
4,CNN1D,ES,ESD,3,0.427924,0.439203,0.011279,0.018134,0.50
5,CNN1D,PES,PESD,3,0.423643,0.405202,-0.018441,0.020128,0.25
6,ResFFN,PS,PSD,3,0.433538,0.430867,-0.002671,0.007600,0.75
7,ResFFN,ES,ESD,3,0.437505,0.445402,0.007897,0.020544,1.00
8,ResFFN,PES,PESD,3,0.465435,0.453887,-0.011548,0.018501,0.50


Cell 13 — Retrain the best configuration and compute bootstrap CI

In [20]:
# Select the best configuration by MEAN Fmax across seeds.

best_row = summary_results.loc[
    summary_results["Validation_best_Fmax_mean"].idxmax()
]

best_architecture = best_row["Architecture"]
best_feature = best_row["Feature"].lower()

print("Best configuration:")
print(best_architecture, "+", best_feature.upper())
print("Mean Fmax:", round(best_row["Overall_Fmax_mean"], 4))

X_best = np.load(
    f"{PROCESSED_DIR}/X_{best_feature}_homology.npy"
).astype(np.float32)

# Retrain seed 42 so one reproducible best-model file is saved.
best_model_path = (
    f"{MODELS_DIR}/best_homology_"
    f"{best_architecture}_{best_feature}_seed42.pt"
)

best_metrics, best_probabilities = train_one_model(
    architecture_name=best_architecture,
    features=X_best,
    seed=42,
    save_checkpoint=True,
    checkpoint_path=best_model_path,
)

best_probabilities = propagate_prediction_scores(best_probabilities)

np.save(
    f"{RESULTS_DIR}/best_model_test_probabilities_homology.npy",
    best_probabilities,
)

best_metadata = {
    "architecture": best_architecture,
    "feature_set": best_feature.upper(),
    "seed": 42,
    "input_dimension": int(X_best.shape[1]),
    "num_go_terms": int(Y_all.shape[1]),
    "metrics": best_metrics,
}

with open(
    f"{MODELS_DIR}/best_homology_model_metadata.json",
    "w",
) as file:
    json.dump(best_metadata, file, indent=2)

print("Saved best model and metadata.")

Best configuration:
MLP + PES
Mean Fmax: 0.4685
Saved best model and metadata.


Cell 14 — Bootstrap CI at the validation-selected threshold

In [21]:
def bootstrap_f_at_fixed_threshold(
    probabilities,
    labels,
    threshold,
    n_resamples=1000,
    seed=0,
):
    """
    Bootstrap test proteins for a 95% CI.

    The threshold was selected from validation data, so it is not optimized
    repeatedly on test resamples.
    """
    rng = np.random.default_rng(seed)

    scores = []
    n_proteins = len(labels)

    for _ in tqdm(
        range(n_resamples),
        desc="Bootstrap resamples",
    ):
        sampled_indices = rng.integers(
            low=0,
            high=n_proteins,
            size=n_proteins,
        )

        score = cafa_f_at_threshold(
            probabilities[sampled_indices],
            labels[sampled_indices],
            threshold,
        )

        scores.append(score)

    scores = np.array(scores)

    return {
        "point_estimate": cafa_f_at_threshold(
            probabilities,
            labels,
            threshold,
        ),
        "CI95_low": float(np.percentile(scores, 2.5)),
        "CI95_high": float(np.percentile(scores, 97.5)),
        "n_resamples": int(n_resamples),
        "threshold_selected_on_validation": float(threshold),
    }


bootstrap_result = bootstrap_f_at_fixed_threshold(
    probabilities=best_probabilities[:, scored_term_mask],
    labels=Y_all[test_idx][:, scored_term_mask],
    threshold=best_metrics["Validation_selected_threshold"],
    n_resamples=cfg["bootstrap"]["n_resamples"],
    seed=cfg["bootstrap"]["random_seed"],
)

bootstrap_result["Architecture"] = best_architecture
bootstrap_result["Feature"] = best_feature.upper()
bootstrap_result["seed"] = 42

with open(
    f"{RESULTS_DIR}/best_model_bootstrap_ci_homology.json",
    "w",
) as file:
    json.dump(bootstrap_result, file, indent=2)

print("Bootstrap result:")
print(bootstrap_result)

Bootstrap resamples:   0%|          | 0/1000 [00:00<?, ?it/s]

Bootstrap result:
{'point_estimate': 0.4406122802122978, 'CI95_low': 0.40993491405659943, 'CI95_high': 0.46965011014814845, 'n_resamples': 1000, 'threshold_selected_on_validation': 0.52, 'Architecture': 'MLP', 'Feature': 'PES', 'seed': 42}


Cell 15 — Final summary

In [22]:
print("=" * 70)
print("TRAINING AND EVALUATION COMPLETE")
print("=" * 70)

print("Completed runs:", len(per_seed_results))
print(
    "Expected runs:",
    9 * 3 * len(cfg["training_seeds"]),
)

print("\nMain result file:")
print(f"{RESULTS_DIR}/full_results_homology_mean_sd_ci.csv")

print("\nOther required reviewer-response outputs:")
print("- continuous_frequency_baseline_homology.csv")
print("- per_seed_results_homology.csv")
print("- dynamic_feature_matched_tests_homology.csv")
print("- test_term_prevalence_homology.csv")
print("- best_model_bootstrap_ci_homology.json")

print("\nNext notebook: 04_xai_figures.ipynb")

TRAINING AND EVALUATION COMPLETE
Completed runs: 81
Expected runs: 81

Main result file:
/content/drive/MyDrive/atlas-go-revision/results/tables/full_results_homology_mean_sd_ci.csv

Other required reviewer-response outputs:
- continuous_frequency_baseline_homology.csv
- per_seed_results_homology.csv
- dynamic_feature_matched_tests_homology.csv
- test_term_prevalence_homology.csv
- best_model_bootstrap_ci_homology.json

Next notebook: 04_xai_figures.ipynb
